# Downsample integrated object to 500k cells (backed mode)

Follows the **backed downsampling** pattern from `preprocessing.ipynb` (the density-aware path): load in backed mode and build the downsampled object from **X, obs, var, uns, and selected obsm only** — **obsp is intentionally skipped** so we never materialize the full-neighbors graph (which would cause OOM on large objects). The simple breast path there uses a full in-memory load then subset; this notebook uses the memory-safe backed build instead.

In [ ]:
import os
import scanpy as sc
import numpy as np

In [ ]:
# Locate repository root and import path configuration
import os
import sys
from pathlib import Path

def _find_repo():
    env = os.environ.get("ATLAS_PIPELINE_ROOT", "").strip()
    if env:
        p = Path(env).expanduser().resolve()
        if (p / "pipeline_paths.py").is_file():
            return p
    cwd = Path.cwd().resolve()
    for d in (cwd, cwd.parent):
        if (d / "pipeline_paths.py").is_file():
            return d
    raise RuntimeError(
        "Set ATLAS_PIPELINE_ROOT to the repository root (folder containing pipeline_paths.py), "
        "or start Jupyter from that directory."
    )

REPO = _find_repo()
if str(REPO) not in sys.path:
    sys.path.insert(0, str(REPO))
import pipeline_paths as pp


**Optional (if you hit MemoryError):** Create a slim copy of the h5ad that keeps only the embeddings you need and **omits obsp** (the huge neighbors graph). Then point `INPUT_PATH` below at the slim file and run the rest of the notebook. Run the cell below once, then set `INPUT_PATH` to `SLIM_PATH` in the next cell.

In [ ]:
INPUT_PATH = str(pp.path_combined_analyzed())
OUTPUT_PATH = str(pp.path_combined_500k())
N_CELLS = 500_000

# Load in backed mode (X stays on disk; only metadata in memory until we subset)
print("Opening in backed mode (memory-efficient)...")
adata_backed = sc.read_h5ad(INPUT_PATH, backed="r")
print(f"Opened: {adata_backed.n_obs:,} cells, {adata_backed.n_vars:,} genes")

In [ ]:
# Choose indices: random 500k or all if already smaller
n_obs = adata_backed.n_obs
if n_obs > N_CELLS:
    print(f"Subsampling from {n_obs:,} to {N_CELLS:,} cells (reading subset from disk)...")
    rng = np.random.default_rng()
    keep_idx = rng.choice(n_obs, size=N_CELLS, replace=False)
else:
    print(f"Using all {n_obs:,} cells (reading from disk)...")
    keep_idx = np.arange(n_obs)

# Build downsampled AnnData from X, obs, var, uns, obsm only (skip obsp) — same as preprocessing backed path
import anndata as ad
from scipy.sparse import issparse as scipy_issparse

keep_idx_sorted = np.sort(keep_idx)
view = adata_backed[keep_idx_sorted]

X_final = view.X
if scipy_issparse(X_final):
    X_final = X_final.copy()
else:
    X_final = X_final.copy()

adata = ad.AnnData(X=X_final, obs=view.obs.copy(), var=adata_backed.var.copy())
if adata_backed.uns:
    adata.uns = adata_backed.uns.copy()

# Copy only obsm (embeddings); obsp is intentionally omitted to avoid OOM
for key in view.obsm.keys():
    arr = view.obsm[key]
    adata.obsm[key] = arr.copy() if hasattr(arr, "copy") else np.asarray(arr).copy()

adata_backed.file.close()
del adata_backed
print(f"Downsampled shape: {adata.shape} (obsp not copied)")

In [ ]:
# Check available embeddings
print(f"\n  Available embeddings in adata.obsm: {list(adata.obsm.keys())}")

In [ ]:
adata.write_h5ad(OUTPUT_PATH)
print(f"Saved to {OUTPUT_PATH}")
assert os.path.exists(OUTPUT_PATH), "Output file was not created"